<a href="https://colab.research.google.com/github/ANUDHEERPARIMI/PyTorch/blob/PyTorch/CNN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader,Dataset
from sklearn.model_selection import train_test_split

In [ ]:
torch.manual_seed(42)

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("zalando-research/fashionmnist")

print("Path to dataset files:", path)

Path to dataset files: /kaggle/input/fashionmnist


In [ ]:
df=pd.read_csv(f"{path}/fashion-mnist_train.csv")

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [ ]:
df.shape

(60000, 785)

In [ ]:
df.head()

,label,pixel1,pixel2,pixel3,pixel4,pixel5,pixel6,pixel7,pixel8,pixel9,...,pixel775,pixel776,pixel777,pixel778,pixel779,pixel780,pixel781,pixel782,pixel783,pixel784
0,2,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,9,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,6,0,0,0,0,0,0,0,5,0,...,0,0,0,30,43,0,0,0,0,0
3,0,0,0,0,1,2,0,0,0,0,...,3,0,0,0,0,1,0,0,0,0
4,3,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [ ]:
X=df.iloc[:,1:].values
y=df.iloc[:,0].values

In [ ]:
X

array([[0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       ...,
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0]])

In [ ]:
y

array([2, 9, 6, ..., 8, 8, 7])

In [ ]:
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=42)

In [ ]:
X_train

array([[0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       ...,
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 1, 0, 0]])

In [ ]:
X_train=X_train/255.0
X_test=X_test/255.0

In [ ]:
class CustomDataset(Dataset):
  def __init__(self,features,label):
    self.features=torch.tensor(features,dtype=torch.float32).reshape(-1,1,28,28)
    self.label=torch.tensor(label,dtype=torch.long)

  def __len__(self):
    return self.features.shape[0]

  def __getitem__(self,index):
    return self.features[index],self.label[index]


In [ ]:
train_dataset=CustomDataset(X_train,y_train)
test_dataset=CustomDataset(X_test,y_test)

batch_size=32

train_loader=DataLoader(train_dataset,batch_size=batch_size,shuffle=True)
test_loader=DataLoader(test_dataset,batch_size=batch_size,shuffle=False)

In [ ]:
class MyNN(nn.Module):
  def __init__(self,input_features):
    super().__init__()
    self.features=nn.Sequential(
        nn.Conv2d(1,32,kernel_size=3,padding='same'),
        nn.ReLU(),
        nn.BatchNorm2d(32),
        nn.MaxPool2d(kernel_size=2,stride=2),

        nn.Conv2d(32,64,kernel_size=3,padding='same'),
        nn.ReLU(),
        nn.BatchNorm2d(64),
        nn.MaxPool2d(kernel_size=2,stride=2)

    )
    self.classifier=nn.Sequential(
        nn.Flatten(),
        nn.Linear(64*7*7,128),
        nn.ReLU(),
        nn.Dropout(p=0.4),

        nn.Linear(128,64),
        nn.ReLU(),
        nn.Dropout(p=0.4),

        nn.Linear(64,10)


    )
  def forward(self,x):
    x=self.features(x)
    c=self.classifier(x)
    return c


In [ ]:
lr=0.01
epochs=100


In [ ]:
model=MyNN(X_train.shape[1])

In [ ]:
model.to(device)

MyNN(
  (features): Sequential(
    (0): Conv2d(1, 32, kernel_size=(3, 3), stride=(1, 1), padding=same)
    (1): ReLU()
    (2): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (4): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=same)
    (5): ReLU()
    (6): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (7): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (classifier): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): Linear(in_features=3136, out_features=128, bias=True)
    (2): ReLU()
    (3): Dropout(p=0.4, inplace=False)
    (4): Linear(in_features=128, out_features=64, bias=True)
    (5): ReLU()
    (6): Dropout(p=0.4, inplace=False)
    (7): Linear(in_features=64, out_features=10, bias=True)
  )
)

In [ ]:
criterian=nn.CrossEntropyLoss()
optimizer=torch.optim.SGD(model.parameters(),lr=lr,weight_decay=1e-4)

In [ ]:
for epoch in range(epochs):
  ttl=0
  for batch_features,batch_labels in train_loader:

    batch_features,batch_labels=batch_features.to(device),batch_labels.to(device)

    outputs=model(batch_features)

    loss=criterian(outputs,batch_labels)

    optimizer.zero_grad()

    loss.backward()

    optimizer.step()

    ttl+=loss.item()
  print(ttl)

559.9971934780478
486.50328923761845
439.95816347002983
399.1697911247611
368.2136558610946
344.7112652249634
319.81097307801247
302.2648715870455
284.89015407674015
268.5723608005792
253.08770627900958
240.62157259183004
225.7195622017607
212.904635717161
204.82392762927338
194.87139053177088
180.56081858230755
172.21153236785904
165.2284391883295
156.58924911590293
152.1778958083596
144.2225212448975
135.66128558223136
127.28162331471685
120.85624017158989
